
# Entropy Feature Extraction
Extracts 4 entropy measures per JMD oscillatory mode (u1 to u4) from all 7173 decomposed npz files.
Entropy types: Permutation, Sample, Kraskov.
Saves 16 new features to features_entropy.csv for merging with the main feature table.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import digamma, gamma
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT   = Path(r"D:\sop")
DECOMPOSED_DIR = PROJECT_ROOT / "data" / "decomposed"
FEATURES_DIR   = PROJECT_ROOT / "data" / "features"

# subsampling limit for entropy methods
# 300 samples gives stable estimates and keeps sample/kraskov entropy fast
MAX_SAMPLES = 300

print("Ready.")

Ready.


In [2]:
# defining entropy functions


def subsample(x, max_n=MAX_SAMPLES):
    """evenly subsample x to max_n points if needed"""
    if len(x) <= max_n:
        return x
    idx = np.linspace(0, len(x) - 1, max_n, dtype=int)
    return x[idx]


def permutation_entropy(x, order=3, delay=1):
    """
    permutation entropy : measures complexity via ordinal rank patterns
    pure numpy, uses full signal (O(N))
    returns value in [0, 1] normalised by log2(order!)
    """
    import math
    N = len(x)
    patterns = np.array([x[i:i + (order - 1) * delay + 1:delay] for i in range(N - (order - 1) * delay)])
    ranked = np.argsort(patterns, axis=1)
    keys = [tuple(r) for r in ranked]
    counts = {}
    for k in keys:
        counts[k] = counts.get(k, 0) + 1
    n_patterns = len(keys)
    probs = np.array(list(counts.values())) / n_patterns
    h = -np.sum(probs * np.log2(probs + 1e-10))
    h_max = np.log2(math.factorial(order))
    return float(h / h_max) if h_max > 0 else 0.0


def sample_entropy(x, order=2):
    """
    sample entropy : measures regularity via template matching
    vectorised with numpy broadcasting : no python loops (O(N^2) in numpy)
    subsampled to MAX_SAMPLES before computation
    tolerance r = 0.2 * std(x)
    """
    x = subsample(x)
    N = len(x)
    r = 0.2 * np.std(x) + 1e-10

    def _count_matches(m):
        # sliding window templates: shape (N-m+1, m)
        templates = np.lib.stride_tricks.sliding_window_view(x, m)
        # pairwise chebyshev distance via broadcasting: shape (n, n)
        diff = np.abs(templates[:, None, :] - templates[None, :, :]).max(axis=2)
        np.fill_diagonal(diff, np.inf)  # exclude self-matches
        return int(np.sum(diff < r))

    b = _count_matches(order)
    a = _count_matches(order + 1)

    if b == 0:
        return 0.0
    val = -np.log((a + 1e-10) / (b + 1e-10))
    return float(val) if np.isfinite(val) else 0.0


def kraskov_entropy(x, k=5):
    """
    kraskov k-nearest neighbour entropy estimator
    python port of KraskovEntropyV2.m provided by professor
    reference: kraskov et al. phys. rev. e 2004
    subsampled to MAX_SAMPLES before computation
    """
    x = subsample(x)
    X = x.reshape(-1, 1)
    n_samples   = X.shape[0]
    n_variables = X.shape[1]

    if k >= n_samples:
        return 0.0

    nbrs = NearestNeighbors(n_neighbors=k + 1, metric='euclidean', algorithm='auto').fit(X)
    distances, _ = nbrs.kneighbors(X)
    distances = distances[:, 1:]       # remove self-distance : mirrors distEpsilon(:,1)=[] in matlab
    epsilon_k = distances[:, k - 1]   # k-th neighbour distance

    Cd = np.pi ** (n_variables / 2) / gamma(1 + n_variables / 2) / (2 ** n_variables)
    H = (digamma(n_samples)
         - digamma(k)
         + np.log(Cd)
         + (n_variables / n_samples) * np.sum(np.log(2 * epsilon_k + 1e-10)))

    return float(H)


print("Entropy functions defined.")
print("  permutation_entropy : ordinal pattern Shannon entropy (order=3, normalised)")
print("  sample_entropy      : vectorised numpy template matching (order=2, r=0.2*std, subsampled to 300)")
print("  kraskov_entropy     : port of KraskovEntropyV2.m (k=5, subsampled to 300)")

Entropy functions defined.
  permutation_entropy : ordinal pattern Shannon entropy (order=3, normalised)
  sample_entropy      : vectorised numpy template matching (order=2, r=0.2*std, subsampled to 300)
  kraskov_entropy     : port of KraskovEntropyV2.m (k=5, subsampled to 300)


In [3]:
# dry run on one file to verify all 3 entropy functions and check timing

import time

sample_file = sorted((DECOMPOSED_DIR / "absent").glob("*.npz"))[0]
d = np.load(sample_file)
u = d["u"]  # shape (4, N)

print(f"File  : {sample_file.name}")
print(f"Shape : u={u.shape}")
print()

for k in range(4):
    mode = u[k]
    t0 = time.time(); pe = permutation_entropy(mode); t_pe = time.time() - t0
    t0 = time.time(); se = sample_entropy(mode);      t_se = time.time() - t0
    t0 = time.time(); ke = kraskov_entropy(mode);     t_ke = time.time() - t0

    print(f"  u{k+1}:  perm={pe:.4f} ({t_pe:.3f}s)  sample={se:.4f} ({t_se:.3f}s)  kraskov={ke:.4f} ({t_ke:.3f}s)")

total_per_file = 4 * (t_pe + t_se + t_ke)
print()
print(f"Estimated time per file : ~{total_per_file:.2f}s")
print(f"Estimated total (7173)  : ~{7173 * total_per_file / 60:.1f} minutes")

File  : a100_AV_decomposed.npz
Shape : u=(4, 30256)

  u1:  perm=0.6306 (0.079s)  sample=1.2465 (0.017s)  kraskov=-3.3068 (0.009s)
  u2:  perm=0.6943 (0.075s)  sample=0.9663 (0.016s)  kraskov=-3.2215 (0.002s)
  u3:  perm=0.7608 (0.075s)  sample=1.3356 (0.017s)  kraskov=-3.5987 (0.003s)
  u4:  perm=0.9892 (0.067s)  sample=0.2721 (0.016s)  kraskov=-3.8920 (0.003s)

Estimated time per file : ~0.35s
Estimated total (7173)  : ~41.3 minutes


In [4]:
# extracting entropy features for all 7173 files
# 12 features per file: 3 entropy types x 4 modes

rows   = []
errors = []

for cls in ["absent", "present", "unknown"]:
    npz_files = sorted((DECOMPOSED_DIR / cls).glob("*.npz"))
    print(f"Processing {cls}: {len(npz_files)} files")

    for i, fpath in enumerate(npz_files):
        try:
            d   = np.load(fpath)
            u   = d["u"]  # shape (4, N)
            row = {"file": fpath.stem, "class": cls}

            for k in range(u.shape[0]):
                mode   = u[k]
                prefix = f"u{k+1}"
                row[f"{prefix}_perm_entropy"]    = permutation_entropy(mode)
                row[f"{prefix}_sample_entropy"]  = sample_entropy(mode)
                row[f"{prefix}_kraskov_entropy"] = kraskov_entropy(mode)

            rows.append(row)

        except Exception as e:
            errors.append({"file": fpath.stem, "class": cls, "error": str(e)})

        if (i + 1) % 250 == 0:
            print(f"  {i+1}/{len(npz_files)} done")

    print(f"  {len(npz_files)}/{len(npz_files)} done")

print(f"\nTotal extracted : {len(rows)}")
print(f"Errors          : {len(errors)}")
if errors:
    for e in errors[:10]:
        print(f"  {e['class']}/{e['file']}: {e['error']}")

Processing absent: 2391 files
  250/2391 done
  500/2391 done
  750/2391 done
  1000/2391 done
  1250/2391 done
  1500/2391 done
  1750/2391 done
  2000/2391 done
  2250/2391 done
  2391/2391 done
Processing present: 2391 files
  250/2391 done
  500/2391 done
  750/2391 done
  1000/2391 done
  1250/2391 done
  1500/2391 done
  1750/2391 done
  2000/2391 done
  2250/2391 done
  2391/2391 done
Processing unknown: 2391 files
  250/2391 done
  500/2391 done
  750/2391 done
  1000/2391 done
  1250/2391 done
  1500/2391 done
  1750/2391 done
  2000/2391 done
  2250/2391 done
  2391/2391 done

Total extracted : 7173
Errors          : 0


In [5]:
# building entropy feature dataframe and checking for issues

df_entropy = pd.DataFrame(rows)
entropy_cols = [c for c in df_entropy.columns if c not in ["file", "class"]]

nan_count = df_entropy[entropy_cols].isna().sum().sum()
inf_count = np.isinf(df_entropy[entropy_cols].values.astype(float)).sum()

print(f"Shape          : {df_entropy.shape}")
print(f"Entropy columns: {len(entropy_cols)}")
print(f"NaN values     : {nan_count}")
print(f"Inf values     : {inf_count}")
print(f"\nClass counts:")
print(df_entropy["class"].value_counts().to_string())
print(f"\nEntropy columns:")
for c in entropy_cols:
    print(f"  {c}")

Shape          : (7173, 14)
Entropy columns: 12
NaN values     : 0
Inf values     : 0

Class counts:
class
absent     2391
present    2391
unknown    2391

Entropy columns:
  u1_perm_entropy
  u1_sample_entropy
  u1_kraskov_entropy
  u2_perm_entropy
  u2_sample_entropy
  u2_kraskov_entropy
  u3_perm_entropy
  u3_sample_entropy
  u3_kraskov_entropy
  u4_perm_entropy
  u4_sample_entropy
  u4_kraskov_entropy


In [6]:
# quick sanity check : mean values per class for each entropy feature
# if any entropy type shows clear separation, it will be visible here

labelled = df_entropy[df_entropy["class"].isin(["absent", "present"])]
print("Mean entropy values by class (absent vs present):")
print(labelled.groupby("class")[entropy_cols].mean().T.to_string())

Mean entropy values by class (absent vs present):
class                 absent   present
u1_perm_entropy     0.587626  0.599028
u1_sample_entropy   1.585305  1.611440
u1_kraskov_entropy -3.297828 -2.933643
u2_perm_entropy     0.694581  0.702699
u2_sample_entropy   1.356765  1.404208
u2_kraskov_entropy -3.668605 -3.246269
u3_perm_entropy     0.796483  0.797141
u3_sample_entropy   1.117326  1.233012
u3_kraskov_entropy -4.047144 -3.533896
u4_perm_entropy     0.900503  0.896977
u4_sample_entropy   0.922719  1.094200
u4_kraskov_entropy -4.426369 -3.857600


In [7]:
# merging entropy features with the existing features.csv
# joining on file column to ensure correct alignment

df_existing = pd.read_csv(FEATURES_DIR / "features.csv")
print(f"Existing features shape : {df_existing.shape}")
print(f"Entropy features shape  : {df_entropy.shape}")

# merging on file
df_merged = df_existing.merge(df_entropy[["file"] + entropy_cols], on="file", how="left")

print(f"Merged shape            : {df_merged.shape}")
print(f"Total feature columns   : {len([c for c in df_merged.columns if c not in ['file','class','label']])}")
print(f"\nNaN after merge: {df_merged[entropy_cols].isna().sum().sum()}")

Existing features shape : (7173, 63)
Entropy features shape  : (7173, 14)
Merged shape            : (7173, 75)
Total feature columns   : 72

NaN after merge: 0


In [8]:
# saving outputs
# features_entropy.csv  : only entropy features + file identifier (for reference)
# features.csv          : full updated table with all 76 features (60 original + 16 entropy)

entropy_out = FEATURES_DIR / "features_entropy.csv"
full_out    = FEATURES_DIR / "features.csv"

df_entropy.to_csv(entropy_out, index=False)
df_merged.to_csv(full_out, index=False)

print(f"Saved: {entropy_out}")
print(f"  rows: {len(df_entropy)}  cols: {df_entropy.shape[1]}  (file + class + 16 entropy features)")
print()
print(f"Saved: {full_out}")
print(f"  rows: {len(df_merged)}  cols: {df_merged.shape[1]}  (3 meta + 60 original + 16 entropy)")
print(f"  size: {full_out.stat().st_size / 1024 / 1024:.1f} MB")

Saved: D:\sop\data\features\features_entropy.csv
  rows: 7173  cols: 14  (file + class + 16 entropy features)

Saved: D:\sop\data\features\features.csv
  rows: 7173  cols: 75  (3 meta + 60 original + 16 entropy)
  size: 9.5 MB


In [9]:
# final verification : reload and confirm

df_check = pd.read_csv(full_out)
all_feat  = [c for c in df_check.columns if c not in ["file", "class", "label"]]
orig_feat = [c for c in all_feat if "entropy" not in c]
entr_feat = [c for c in all_feat if "entropy" in c]

print(f"Reloaded shape     : {df_check.shape}")
print(f"Original features  : {len(orig_feat)}")
print(f"Entropy features   : {len(entr_feat)}")
print(f"Total features     : {len(all_feat)}")
print(f"NaN count          : {df_check[all_feat].isna().sum().sum()}")
print(f"Classes            : {df_check['class'].value_counts().to_dict()}")

Reloaded shape     : (7173, 75)
Original features  : 56
Entropy features   : 16
Total features     : 72
NaN count          : 0
Classes            : {'absent': 2391, 'present': 2391, 'unknown': 2391}
